# Z nagrania USG do masy płodu
### Sympozjum DIL, Wrocław, 28.11.2026 · Kamil Jędryczek

> **Materiał edukacyjny. Nie jest wyrobem medycznym i nie służy do decyzji klinicznych.**

Zadajemy dwa pytania:

1. **„Ile waży teraz?”** Model sam przegląda całe nagranie USG (na żywo co 4. klatkę, w wynikach przygotowanych wcześniej co 2.), znajduje klatkę z brzuchem płodu, mierzy obwód brzucha (AC) i z tej jednej miary szacuje dzisiejszą masę. Robią to dwa modele: holenderski nnU-Net z konkursu ACOUSLIC-AI i polski FUVAI (Sano + Warszawski Uniwersytet Medyczny), który ma publiczne wagi.
2. **„Ile waży w dniu porodu?”** Na to odpowiada polski model z Sano w Krakowie (model bazowy z pracy TabAttention, MICCAI 2023): ogląda 16 klatek brzucha i szacuje masę w dniu porodu. Uczono go wyłącznie na badaniach wykonanych mniej niż 24 h przed porodem. Uruchamiamy go z publicznie udostępnionego checkpointu modelu bazowego (obrazowego). Nasze nagrania to ok. 28–34 tc., więc model wychodzi poza dane, na których się uczył: zobaczymy, czego nie wie.

Uczciwie od początku: **to nie jest narzędzie lepsze od Hadlocka** (takiego dziś publicznie nie ma). Mocna strona tej ścieżki to pełna automatyzacja na trudnych danych: tania głowica, smartfon, nagranie nowicjusza „na ślepo”.

**Dane:** ACOUSLIC-AI (Radboudumc, Zenodo 12697994), licencja CC BY-NC-SA 4.0, tylko do użytku niekomercyjnego.

**Jak uruchomić:** menu `Run` → `Run All Cells`. Bez internetu działa tryb `"z_pamieci"`: wyniki policzone wcześniej skryptem `scripts/przygotuj_demo.py`.

In [ ]:
# ── KONFIGURACJA ──────────────────────────────────────────────
TRYB = "z_pamieci"        # "na_zywo" = liczymy model teraz; "z_pamieci" = wyniki policzone wcześniej (bez internetu)
NR_NAGRANIA = 1           # 0 = wycinek startowy (128 klatek), 1+ = pełne nagrania z Zenodo (840 klatek)

# Co którą klatkę model ogląda na żywo. Zmierzone na M1 Pro (MPS), nagranie 840 klatek,
# czarne klatki pominięte: co 4. → 210 klatek, nnU-Net ok. 25–35 s, FUVAI ok. 8 s
# (wycinek 128 klatek: FUVAI ok. 2–3 s);
# co 8. → mniej więcej połowa czasu. Do tego 3–5 s na wczytanie modeli. Na wolniejszym komputerze wpisz 8.
CO_KTORA_NA_ZYWO = 4

import os
from pathlib import Path

# nnU-Net bez tych zmiennych drukuje ostrzeżenia; my ich nie potrzebujemy (model czytamy z folderu).
for zmienna in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):
    os.environ.setdefault(zmienna, str(Path("../pamiec").resolve()))
os.environ.setdefault("nnUNet_n_proc_DA", "12")   # bez tego nnU-Net przy imporcie uruchamia program `hostname` (dotyczy tylko treningu)

import time
import numpy as np, matplotlib.pyplot as plt
from usg_masa import biometria, dane, segmentacja as seg
from usg_masa import zasoby

def odmiana(n, jeden, kilka, wiele):
    """Polska liczba mnoga: 1 klatka, 2–4 klatki, 5+ klatek (12–14 klatek)."""
    if n == 1:
        return jeden
    return kilka if n % 10 in (2, 3, 4) and n % 100 not in (12, 13, 14) else wiele

def po_polsku(x, miejsca=1):
    """Liczba z przecinkiem dziesiętnym."""
    return f"{x:.{miejsca}f}".replace(".", ",")

sciezki = [dane.pobierz_przypadek_startowy()] + sorted(
    p for p in dane.KATALOG_DANYCH.rglob("*.mha") if "masks" not in p.parts)   # bez plików masek
CSV = dane.KATALOG_DANYCH / "zenodo" / dane.CSV_OBWODOW
REFERENCJA = dane.wczytaj_obwody_referencyjne(CSV) if CSV.exists() else None
print("Dostępne nagrania:")
for nr, p in enumerate(sciezki):
    print(f"  {nr}: {p.stem}")
if NR_NAGRANIA >= len(sciezki):
    print(f"Nie ma nagrania nr {NR_NAGRANIA} — biorę wycinek startowy (nr 0).")
    NR_NAGRANIA = 0
SWEEP = dane.wczytaj_mha(sciezki[NR_NAGRANIA])
n = len(SWEEP.klatki)
print(f"\nWybrane: {SWEEP.nazwa}: {n} {odmiana(n, 'klatka', 'klatki', 'klatek')}, "
      f"piksel {po_polsku(SWEEP.odstep_mm, 2)} mm")

## Nagranie

Tak wygląda materiał, który dostaje model:

- Pełne nagranie to **840 klatek = 6 przeciągnięć głowicą „na ślepo”, po 140 klatek**. Tak wygląda protokół ACOUSLIC-AI: tania głowica podłączona do smartfona, a badanie wykonują osoby bez doświadczenia w USG, według prostej instrukcji, bez szukania konkretnego przekroju.
- Plik startowy (nagranie nr 0) to tylko **128-klatkowy wycinek** jednego nagrania.
- Całkiem czarne klatki na końcu każdego przeciągnięcia to puste dopełnienie nagrania; model je pomija.
- Nikt nie zaznaczył, gdzie jest brzuch. **Model musi sam go znaleźć.**

Przesuń suwak, żeby obejrzeć kolejne klatki.

In [ ]:
# ── NAGRANIE ──────────────────────────────────────────────────
import ipywidgets as widgets

def pokaz_klatke(i):
    plt.figure(figsize=(7, 5.5))
    plt.imshow(SWEEP.klatki[i], cmap="gray")
    plt.axis("off")
    opis = f"Klatka {i} z {len(SWEEP.klatki)}"
    if len(SWEEP.klatki) == 840:
        opis += f" · przeciągnięcie głowicą nr {i // 140 + 1} z 6"
    plt.title(opis)
    plt.gca().text(0.5, -0.01, dane.ATRYBUCJA, transform=plt.gca().transAxes, ha="center", va="top", fontsize=8)
    plt.show()

widgets.interact(pokaz_klatke, i=widgets.IntSlider(
    value=len(SWEEP.klatki) // 2, min=0, max=len(SWEEP.klatki) - 1, step=1,
    description="klatka", continuous_update=False, layout=widgets.Layout(width="90%")));

## Model szuka brzucha

Model segmentacji (nnU-Net z rozwiązania bazowego konkursu ACOUSLIC-AI) przegląda całe nagranie: na żywo co 4. klatkę (ustawienie `CO_KTORA_NA_ZYWO`), w wynikach przygotowanych wcześniej co 2. W każdej oglądanej klatce zaznacza piksele brzucha płodu. Rozróżnia dwa rodzaje przekroju:

- **płaszczyzna optymalna**: przekrój, na którym lekarz zmierzyłby obwód brzucha,
- **suboptymalna**: brzuch widać, ale przekrój jest skośny.

Wykres pokazuje, ile pikseli brzucha model widział w każdej klatce. Wybieramy klatkę z największym polem.

In [ ]:
# ── MODEL SZUKA BRZUCHA W CAŁYM NAGRANIU ──────────────────────
if TRYB == "na_zywo":
    import contextlib, io
    start = time.perf_counter()
    with contextlib.redirect_stdout(io.StringIO()):   # chowamy techniczne komunikaty nnU-Netu
        predyktor = seg.zaladuj_predyktor(zasoby.katalog_modelu_nnunet())
    plt.switch_backend("module://matplotlib_inline.backend_inline")   # nnU-Net wyłącza rysowanie w notebooku; włączamy z powrotem
    print(f"Model wczytany w {po_polsku(time.perf_counter() - start)} s")
    WYNIK = seg.segmentuj_sweep_szybko(predyktor, SWEEP, co_ktora=CO_KTORA_NA_ZYWO)   # skrót: bez kafli nnU-Netu
    skad = "teraz"
else:
    plik = seg.KATALOG_PAMIECI / f"{SWEEP.nazwa}.npz"
    if not plik.exists():
        raise FileNotFoundError(f"Brak {plik.name} w pamięci. Uruchom: uv run python scripts/przygotuj_demo.py "
                                f"albo ustaw TRYB = \"na_zywo\".")
    WYNIK = seg.wczytaj_wynik(plik)
    skad = "wcześniej, pełnym nnU-Netem"
n = len(WYNIK.indeksy)
krok = int(WYNIK.indeksy[1] - WYNIK.indeksy[0]) if n > 1 else 1
czarne = int(sum(SWEEP.klatki[i].max() == 0 for i in WYNIK.indeksy))
print(f"Przeanalizowano {n} {odmiana(n, 'klatkę', 'klatki', 'klatek')} (co {krok}. z {len(SWEEP.klatki)}) "
      f"w {WYNIK.czas_s:.0f} s ({skad}); najlepsza klatka: {WYNIK.najlepsza_klatka}")
if czarne:
    print(f"Wśród nich całkiem czarnych (pusty koniec przeciągnięcia): {czarne}. Model ich nie ogląda, liczy je jako tło.")

plt.figure(figsize=(10, 3))
plt.plot(WYNIK.indeksy, WYNIK.pole_opt, label="płaszczyzna optymalna")
plt.plot(WYNIK.indeksy, WYNIK.pole_subopt, label="suboptymalna")
if WYNIK.najlepsza_klatka >= 0:
    plt.axvline(WYNIK.najlepsza_klatka, color="black", ls=":", label="wybrana klatka")
if len(SWEEP.klatki) == 840:
    for granica in range(140, 840, 140):
        plt.axvline(granica, color="gray", lw=0.5)   # granice przeciągnięć głowicą
plt.xlabel("numer klatki"); plt.ylabel("pole brzucha [px]"); plt.legend(loc="upper right"); plt.show()

## Pomiar i masa teraz

To jest masa ***dziś***, z jednej miary (AC), wzorem Campbella i Wilkina (1975). Obok wzorem Hadlocka (1984) podajemy wiek ciążowy odpowiadający temu AC („tc.”): w którym tygodniu przeciętny płód ma taki obwód brzucha. To nie jest datowanie tej ciąży. Jedna miara to przybliżenie: pełna biometria (HC, BPD, FL) jest dokładniejsza.

Lekarz robi to ręcznie, na wybranej przez siebie klatce; tu model sam wybrał klatkę z całego nagrania. Obok: pomiar ekspertów ACOUSLIC-AI do porównania.

Jak w rozwiązaniu bazowym ACOUSLIC-AI bierzemy klatkę z największym polem brzucha. Często jest to przekrój skośny (suboptymalny), który może zawyżać AC. Mimo to na trzech z czterech przygotowanych nagrań AC wychodzi w granicach ok. 10 mm od pomiaru ekspertów; na nagraniu nr 3 jest o ok. 25 mm za duże.

Czerwony obrys to maska z modelu, żółta przerywana linia to dopasowana elipsa. Z jej obwodu liczymy AC.

In [ ]:
# ── POMIAR I MASA TERAZ ───────────────────────────────────────
import cv2
from matplotlib.patches import Ellipse

o = biometria.oszacuj(WYNIK.maska, SWEEP.odstep_mm)
if WYNIK.najlepsza_klatka >= 0 and WYNIK.maska is not None:
    klatka = SWEEP.klatki[WYNIK.najlepsza_klatka]
    plt.figure(figsize=(8, 6.5))
    plt.imshow(klatka, cmap="gray")
    plt.contour(WYNIK.maska, levels=[0.5], colors="red", linewidths=1.5)
    if o.ac_mm is not None:   # elipsę rysujemy tylko wtedy, gdy dało się ją dopasować
        kontury, _ = cv2.findContours(WYNIK.maska.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        (x, y), (d1, d2), kat = cv2.fitEllipse(max(kontury, key=cv2.contourArea))
        plt.gca().add_patch(Ellipse((x, y), d1, d2, angle=kat, fill=False, color="yellow", ls="--", lw=1.5))
    plt.xlim(0, klatka.shape[1]); plt.ylim(klatka.shape[0], 0)
    plt.axis("off")
    plt.title(f"Klatka {WYNIK.najlepsza_klatka}: brzuch płodu wybrany przez model")
    plt.gca().text(0.5, -0.01, dane.ATRYBUCJA, transform=plt.gca().transAxes, ha="center", va="top", fontsize=8)
    plt.show()
print(o.komunikat)
if WYNIK.najlepsza_klatka >= 0:
    i = int(np.flatnonzero(WYNIK.indeksy == WYNIK.najlepsza_klatka)[0])
    plaszczyzna = "optymalna" if WYNIK.pole_opt[i] >= WYNIK.pole_subopt[i] else "suboptymalna (skośna)"
    print(f"Płaszczyzna wybranej klatki wg modelu: {plaszczyzna}.")

if REFERENCJA is not None and dane.ac_referencyjne(REFERENCJA, SWEEP.nazwa):
    eksperci = dane.ac_referencyjne(REFERENCJA, SWEEP.nazwa)
    print(f"AC zmierzone przez ekspertów ACOUSLIC-AI: {', '.join(f'{ac:.0f}' for ac in eksperci)} mm")
    if len(SWEEP.klatki) == 840 and WYNIK.najlepsza_klatka >= 0:
        nr = WYNIK.najlepsza_klatka // 140 + 1
        ekspert = REFERENCJA.loc[SWEEP.nazwa.split("_")[0], f"sweep_{nr}_ac_mm"]
        print(f"Wybrana klatka pochodzi z przeciągnięcia głowicą nr {nr} z 6; " +
              (f"ekspert zmierzył w nim {ekspert:.0f} mm." if ekspert == ekspert   # NaN ≠ NaN
               else "w tym przeciągnięciu ekspert nie mierzył."))

## Opcja pośrednia: polski FUVAI (Sano + WUM, 2022)

FUVAI (Płotka i in., *Physics in Medicine & Biology* 2022) powstał w Sano w Krakowie razem z Warszawskim Uniwersytetem Medycznym. Jedna sieć robi dwie rzeczy naraz:

- **rozpoznaje, co widać w klatce**: głowę, brzuch, kość udową albo tło,
- **zaznacza tę strukturę** na obrazie; z obrysu brzucha liczymy AC tak samo jak wyżej.

Uczono go na nagraniach z 700 ciąż z WUM (aparaty GE), robionych „z ręki” w płaszczyznach standardowych. W artykule na 50 nagraniach mierzył porównywalnie z 5 doświadczonymi ultrasonografistami. **W odróżnieniu od BabyNet++ autorzy opublikowali wagi**, więc możemy go uruchomić.

Uczciwie, zanim spojrzymy na wynik:

- **Nasze nagrania są poza jego domeną.** Autorzy biorą do pomiaru tylko klatki, w których model jest pewny brzucha na ponad 0,9. Na nagraniach „na ślepo” z ACOUSLIC-AI prawie się to nie zdarza: w czterech nagraniach próg przekroczyła jedna klatka (gdy oglądamy co 2. klatkę; co 4. żadna).
- **Wybór klatki to nasza adaptacja**, nie metoda autorów: bierzemy 10% klatek, w których model najbardziej „wierzy” w brzuch (bywa, że ta wiara to kilka procent). Z nich zostają tylko te, w których brzuch jest dla modelu najbardziej prawdopodobny (a nie głowa, kość udowa czy tło). Spośród nich wygrywa ta, której obrys najlepiej pasuje do elipsy. Gdy takiej klatki nie ma, mówimy wprost: „nie znaleziono”.
- **Nie mamy pewności, że wybrana klatka to brzuch.** Na nagraniach 1 i 2 nnU-Net w klatce wybranej przez FUVAI nie widzi brzucha wcale, a na nagraniu 2 FUVAI waha się między brzuchem (0,53) a głową (0,38). Dlatego poniżej pokazujemy, co FUVAI „widzi” w wybranej klatce.
- Każdą klatkę model ogląda osobno. Sieć umie oglądać serię kolejnych klatek, ale na tych nagraniach seria pogarsza wynik.
- **Masę liczymy z samego AC, wzorem Campbella**, jak wyżej. Autorzy FUVAI liczą ją wzorem Hadlocka z HC, AC i FL, a gdy brakuje którejkolwiek z trzech płaszczyzn (głowa, brzuch, kość udowa), masy w ogóle nie podają. W ACOUSLIC-AI jest tylko brzuch, więc FUVAI w wersji z artykułu nie dałby tu masy wcale.
- Gdy FUVAI nie jest pewny brzucha (poniżej 0,9), a nnU-Net w tej samej klatce brzucha nie widzi, pokazujemy obrys i AC, ale masy nie podajemy.
- Wynik zależy od tego, co którą klatkę oglądamy. Na nagraniach 1–3 co 2. klatka daje AC 242 / 273 / 209 mm, a co 4. klatka 242 mm / nic / 204 mm: na nagraniu 2 FUVAI nie znajduje wtedy żadnej klatki brzucha. Na wycinku (nagranie 0) nie znajduje nic w obu ustawieniach.
- Kod autorów nie ma licencji: pobieramy go z GitHuba na swój komputer, nie rozpowszechniamy.

Na wykresie: pewność modelu, że w klatce jest brzuch, i próg 0,9 z artykułu. Na zdjęciu: obrys FUVAI na żółto, obrys nnU-Netu z poprzedniego kroku na czerwono (nnU-Net często wybiera inną klatkę).

In [ ]:
# ── FUVAI: POLSKI MODEL Z PUBLICZNYMI WAGAMI ──────────────────
from matplotlib.lines import Line2D
from usg_masa import fuvai

if TRYB == "na_zywo":
    start = time.perf_counter()
    MODEL_FUVAI = fuvai.zbuduj_fuvai()
    print(f"Model FUVAI wczytany w {po_polsku(time.perf_counter() - start)} s")
    import warnings
    with warnings.catch_warnings():   # chowamy ostrzeżenia PyTorcha o przestarzałych funkcjach w kodzie autorów
        warnings.simplefilter("ignore", UserWarning)
        WYNIK_FUVAI = fuvai.analizuj_sweep(MODEL_FUVAI, SWEEP, co_ktora=CO_KTORA_NA_ZYWO)
    skad_fuvai = "teraz"
else:
    plik = seg.KATALOG_PAMIECI / f"{SWEEP.nazwa}_fuvai.npz"
    if not plik.exists():
        raise FileNotFoundError(f"Brak {plik.name} w pamięci. Uruchom: uv run python scripts/przygotuj_demo.py "
                                f"albo ustaw TRYB = \"na_zywo\".")
    WYNIK_FUVAI = fuvai.wczytaj_wynik(plik)
    skad_fuvai = "wcześniej"
F = WYNIK_FUVAI
p_brzuch = F.p_klas[:, fuvai.BRZUCH]
ile = len(F.indeksy)
co_ile = int(F.indeksy[1] - F.indeksy[0]) if ile > 1 else 1
ocenione = np.array([SWEEP.klatki[i].max() > 0 for i in F.indeksy])   # czarnych klatek model nie ogląda
print(f"FUVAI przeanalizował {ile} {odmiana(ile, 'klatkę', 'klatki', 'klatek')} (co {co_ile}. z {len(SWEEP.klatki)}) "
      f"w {F.czas_s:.0f} s ({skad_fuvai}).")
if ocenione.any():
    prog_10 = np.quantile(p_brzuch[ocenione], 1 - fuvai.GORNE_PROCENT / 100)
    print(f"Najwyższa pewność, że w klatce jest brzuch: {po_polsku(p_brzuch.max(), 2)} "
          f"(próg z artykułu: {po_polsku(fuvai.PROG_ARTYKULU)}). "
          f"Kandydaci do pomiaru (górne {fuvai.GORNE_PROCENT}%): pewność od {po_polsku(prog_10, 2)} w górę.")

plt.figure(figsize=(10, 3))
plt.plot(F.indeksy, p_brzuch, label="p(brzuch) wg FUVAI")
plt.axhline(fuvai.PROG_ARTYKULU, color="gray", ls="--", label="próg z artykułu")
if F.najlepsza_klatka >= 0:
    plt.axvline(F.najlepsza_klatka, color="goldenrod", ls=":", lw=2, label=f"klatka FUVAI ({F.najlepsza_klatka})")
if WYNIK.najlepsza_klatka >= 0:
    plt.axvline(WYNIK.najlepsza_klatka, color="red", ls=":", label=f"klatka nnU-Net ({WYNIK.najlepsza_klatka})")
if len(SWEEP.klatki) == 840:
    for granica in range(140, 840, 140):
        plt.axvline(granica, color="gray", lw=0.5)   # granice przeciągnięć głowicą
plt.ylim(0, 1.02)
plt.xlabel("numer klatki"); plt.ylabel("p(brzuch)"); plt.legend(loc="upper right", fontsize=8); plt.show()

def obrysy(os_, nr_klatki, maski, tytul):
    os_.imshow(SWEEP.klatki[nr_klatki], cmap="gray")
    for maska, kolor in maski:
        os_.contour(maska, levels=[0.5], colors=kolor, linewidths=1.5)
    os_.set_title(tytul, fontsize=10); os_.axis("off")

if F.najlepsza_klatka >= 0 and F.maska is not None:
    o_fuvai = biometria.oszacuj(F.maska, SWEEP.odstep_mm)
    jest_nnunet = WYNIK.najlepsza_klatka >= 0 and WYNIK.maska is not None
    legenda = [Line2D([], [], color="yellow", lw=2, label=f"FUVAI, klatka {F.najlepsza_klatka}")]
    if jest_nnunet and WYNIK.najlepsza_klatka == F.najlepsza_klatka:
        fig, os_ = plt.subplots(figsize=(8, 6.5))
        obrysy(os_, F.najlepsza_klatka, [(F.maska, "yellow"), (WYNIK.maska, "red")],
               f"Klatka {F.najlepsza_klatka}: FUVAI i nnU-Net wybrały tę samą klatkę")
        legenda.append(Line2D([], [], color="red", lw=2, label=f"nnU-Net, klatka {WYNIK.najlepsza_klatka}"))
        os_.legend(handles=legenda, loc="upper right", fontsize=9)
    elif jest_nnunet:
        fig, osie = plt.subplots(1, 2, figsize=(13, 5.5))
        obrysy(osie[0], F.najlepsza_klatka, [(F.maska, "yellow")], f"FUVAI (żółty): klatka {F.najlepsza_klatka}")
        obrysy(osie[1], WYNIK.najlepsza_klatka, [(WYNIK.maska, "red")], f"nnU-Net (czerwony): klatka {WYNIK.najlepsza_klatka}")
    else:
        fig, os_ = plt.subplots(figsize=(8, 6.5))
        obrysy(os_, F.najlepsza_klatka, [(F.maska, "yellow")], f"FUVAI (żółty): klatka {F.najlepsza_klatka}")
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    fig.text(0.5, 0.015, dane.ATRYBUCJA, ha="center", fontsize=8)
    plt.show()

    # Czy nnU-Net widzi brzuch w klatce wybranej przez FUVAI? (None = nnU-Net tej klatki nie oglądał)
    if jest_nnunet and F.najlepsza_klatka == WYNIK.najlepsza_klatka:
        pole_nnunet = int(np.count_nonzero(WYNIK.maska))
    elif F.najlepsza_klatka in WYNIK.indeksy:
        j = int(np.flatnonzero(WYNIK.indeksy == F.najlepsza_klatka)[0])
        pole_nnunet = int(WYNIK.pole_opt[j] + WYNIK.pole_subopt[j])
    else:
        pole_nnunet = None
    bez_masy = F.p_brzuch_najlepszej < fuvai.PROG_ARTYKULU and pole_nnunet == 0

    print("Obwód brzucha (AC) i masa z samego AC (wzór Campbella):")
    if bez_masy and o_fuvai.ac_mm is not None:
        print(f"  FUVAI (Sano + WUM), klatka {F.najlepsza_klatka}: AC = {o_fuvai.ac_mm:.0f} mm (masy nie podajemy, patrz niżej)")
    else:
        print(f"  FUVAI (Sano + WUM), klatka {F.najlepsza_klatka}: {o_fuvai.komunikat}")
    print(f"  nnU-Net (ACOUSLIC-AI), klatka {WYNIK.najlepsza_klatka}: {o.komunikat}")
    if REFERENCJA is not None and dane.ac_referencyjne(REFERENCJA, SWEEP.nazwa):
        eksperci = dane.ac_referencyjne(REFERENCJA, SWEEP.nazwa)
        print(f"  Eksperci ACOUSLIC-AI: {', '.join(f'{ac:.0f}' for ac in eksperci)} mm")
    p_wybranej = F.p_klas[int(np.flatnonzero(F.indeksy == F.najlepsza_klatka)[0])]
    print("Co FUVAI widzi w tej klatce: " + " · ".join(
        f"{fuvai.KLASY[k]} {po_polsku(p_wybranej[k], 2)}" for k in np.argsort(-p_wybranej)))
    if F.p_brzuch_najlepszej > fuvai.PROG_ARTYKULU:
        print("Pewność brzucha jest powyżej progu 0,9 z artykułu.")
    else:
        print("Pewność brzucha jest poniżej progu 0,9 z artykułu: autorzy nie wzięliby tej klatki.")
    if pole_nnunet is not None and F.najlepsza_klatka != WYNIK.najlepsza_klatka:
        print("W tej samej klatce nnU-Net " + (f"też widzi brzuch ({pole_nnunet} px)." if pole_nnunet else
              "nie widzi brzucha wcale. Nie mamy pewności, że FUVAI zmierzył tu brzuch."))
    if bez_masy:
        print(f"Masy nie podajemy: FUVAI nie jest pewny (p(brzuch) = {po_polsku(F.p_brzuch_najlepszej, 2)} < "
              f"{po_polsku(fuvai.PROG_ARTYKULU)}), a nnU-Net nie widzi w tej klatce brzucha. "
              "To może być inna struktura (np. główka).")
else:
    print(f"FUVAI nie znalazł w tym nagraniu klatki brzucha: najwyższa pewność to {po_polsku(p_brzuch.max(), 2)}, "
          "a wśród klatek-kandydatek nie ma takiej, w której brzuch byłby dla modelu najbardziej prawdopodobny "
          "i dał pełny obrys. Nie podajemy AC ani masy.")

## Polski model: masa w dniu porodu (Sano)

Trzeci krok. Do tej pory: (1) nnU-Net z ACOUSLIC-AI działa, ale to model holenderski; (2) polski FUVAI ma publiczne wagi, ale na tych nagraniach działa tylko częściowo. Oba mierzą masę ***dziś***. Teraz (3) polski model, który z samego obrazu szacuje masę ***w dniu porodu***.

**BabyNet++** (Płotka i in., Sano Kraków, *Computers in Biology and Medicine* 2023) był pierwszym kandydatem: sieć 3D (ResNet-18 z uwagą wielogłowicową) ogląda nagranie klip po klipie, dokłada cechy kliniczne (AC, HC, BPD, FL, wiek matki, wiek ciążowy) i podaje masę urodzeniową. Kod jest publiczny, ale wytrenowane wagi nie są publicznie dostępne.

**Model bazowy Sano** pochodzi z kolejnej pracy tego zespołu: *TabAttention* (Grzeszczyk, Płotka i in., MICCAI 2023; Sano Kraków + WUM). To sieć 3D ResNet-18: ogląda **16 klatek nagrania brzucha** (128×128 pikseli) i zwraca masę w dniu porodu w gramach, z samego obrazu, bez danych klinicznych. Kod i źródła: Grzeszczyk, Płotka i in., MICCAI 2023 (TabAttention), Sano + WUM. Publiczny checkpoint to model bazowy obrazowy; nie redystrybuujemy kodu autorów ani wag.

- **Na czym się uczył:** 92 nagrania brzucha z WUM, wykonane **mniej niż 24 h przed porodem**; ciąże donoszone (38,3 ± 1,5 tc.), masa urodzeniowa 3 495 ± 507 g.
- **Jak trafiał u autorów:** średni błąd 289 g (8,5%), czyli gorzej niż klinicyści z wzorem Hadlocka (205 g, 5,9%). Najlepszy był pełny TabAttention: **obraz + 6 pomiarów biometrycznych lekarza**, 170 g (5,0%), lepiej niż klinicyści. AI nie zastępuje pomiarów lekarza, tylko je wzmacnia. Wag pełnego modelu nie ma.
- **Jak go używamy:** jak autorzy, z dwiema różnicami. Autorzy mieli klatki brzucha zaznaczone ręcznie; my bierzemy klatki, w których brzuch widzi nnU-Net z kroku 1. Tniemy je na segmenty po 16 klatek (co 3. klatka) i zmieniamy rozmiar piksela na 0,2 mm. To drugie to **nasza interpretacja** opisu w pracy (piksel 0,2 mm); kod autorów obrazu tak nie przeskalowuje. Wynik nagrania to średnia z segmentów. Na naszych nagraniach klatek brzucha starcza na **jeden** segment.

Uczciwie, zanim spojrzymy na liczbę:

- **To masa *w dniu porodu*, nie prognoza na tygodnie do przodu.** Model widział wyłącznie badania wykonane mniej niż 24 h przed porodem. Zna tylko jedno pytanie: ile waży dziecko, które urodzi się w ciągu doby?
- **Nasze nagrania to ok. 28–34 tc.** (wiek wg AC z kroku 1). Na nagraniach z 28.–34. tc. model **wychodzi poza dane, na których się uczył** — i na wszystkich czterech podaje liczbę typową dla noworodka donoszonego, **3,6–4,0 kg**, bez cienia wątpliwości, choć dziś te płody ważą wg AC ok. 1,4–2,6 kg. Wynik niewiele zależy od tego, co model widzi: gdy zamiast klatek brzucha damy mu wszystkie niepuste klatki nagrania, też wychodzi 3,6–3,9 kg.
- **To ekstrapolacja, nie „prognoza na termin”.** Model nigdy nie widział ciąży w 30. tygodniu i nie uczył się, jak płód rośnie do porodu. Mas urodzeniowych tych dzieci zresztą nie znamy (ACOUSLIC-AI ich nie podaje), więc nie ma tu żadnej walidacji.
- **Wynik jest kruchy.** Bez zmiany rozmiaru piksela (0,28 → 0,2 mm) ten sam model na tych samych klatkach daje 3,0–3,6 kg. Na żywo nnU-Net ogląda co 4. klatkę, więc model Sano dostaje trochę inne klatki brzucha (często za mało na pełny segment) i wynik przesuwa się o 30–320 g. A nasz „segment” to klatki z kilku przeciągnięć głowicą „na ślepo”, a nie ciągły, 5–10-sekundowy film w płaszczyźnie standardowej jak u autorów (numery klatek nad obrazkami poniżej).
- **Model nie wie, kiedy nie wie.** Na każde nagranie odpowie liczbą w gramach, z tą samą pewnością siebie, także daleko poza tym, czego się uczył. To samo zjawisko znamy z modeli językowych (halucynacje). Pytanie do każdego produktu AI: „na jakich danych to uczono?”.
- Kod autorów nie ma licencji: pobieramy go z GitHuba na swój komputer; kodu ani wag nie rozpowszechniamy.

Poniżej: 16 klatek, które dostaje model (już w jego rozdzielczości), i porównanie dzisiejszej masy z wynikiem modelu (masa w dniu porodu) na tle tego, na czym model się uczył.

In [ ]:
# ── MODEL SANO: CAŁE NAGRANIE → MASA W DNIU PORODU ──────────
from usg_masa import model_sano

MASA_UR_UCZACA, SD_UCZACA = 3495, 507   # masa urodzeniowa w zbiorze uczącym modelu (praca TabAttention): średnia ± SD

def gramy(x):
    """Masa z odstępem tysięcy: 3551,2 → „3 551”."""
    return f"{x:,.0f}".replace(",", " ")

if TRYB == "na_zywo":
    start = time.perf_counter()
    MODEL_SANO = model_sano.zbuduj_model_sano()   # 3D ResNet-18 z publicznego checkpointu modelu bazowego, na procesorze
    print(f"Model Sano wczytany w {po_polsku(time.perf_counter() - start)} s")
    WYNIK_SANO = model_sano.przewidz(MODEL_SANO, SWEEP, WYNIK)   # klatki brzucha wg nnU-Netu → segmenty → średnia
    skad_sano = "policzone teraz, na procesorze"
else:
    plik = seg.KATALOG_PAMIECI / f"{SWEEP.nazwa}_sano.json"
    if not plik.exists():
        raise FileNotFoundError(f"Brak {plik.name} w pamięci. Uruchom: uv run python scripts/przygotuj_demo.py "
                                f"albo ustaw TRYB = \"na_zywo\".")
    WYNIK_SANO = model_sano.wczytaj_wynik(plik)
    skad_sano = "wynik policzony wcześniej"
S = WYNIK_SANO

# Te same klatki, które dostał model: klatki brzucha wg nnU-Netu, pierwszy segment (16 klatek, co 3.)
segm, _ = model_sano.segmenty(model_sano.klatki_brzucha(SWEEP, WYNIK))
if segm:
    pierwszy = segm[0]
    klip = model_sano.klip_tensor([SWEEP.klatki[j] for j in pierwszy], SWEEP.odstep_mm)[0, 0].numpy()
    fig, osie = plt.subplots(2, 8, figsize=(12, 3.6))
    for k, os_ in enumerate(osie.flat):
        os_.imshow(klip[k], cmap="gray", vmin=-1, vmax=1); os_.axis("off")
        os_.set_title(f"klatka {pierwszy[k]}" if k < len(pierwszy) else "dopełnienie", fontsize=8)
    fig.suptitle("Tak widzi nagranie model Sano: pierwszy segment klatek brzucha (wg nnU-Netu), 128×128 pikseli (obraz zmniejszony jak u autorów)",
                 fontsize=11)
    fig.text(0.5, 0.01, dane.ATRYBUCJA, ha="center", fontsize=8)
    plt.show()

n_b, n_s = S.n_klatek_brzucha, len(S.oceny_g)
if S.masa_g is None:
    print("nnU-Net nie widzi w tym nagraniu brzucha, więc model Sano nie ma czego oglądać. Wyniku nie podajemy.")
else:
    print(f"Klatek brzucha wg nnU-Netu: {n_b}. Model Sano złożył z nich {n_s} "
          f"{odmiana(n_s, 'segment', 'segmenty', 'segmentów')} po 16 klatek i policzył "
          f"w {po_polsku(S.czas_s)} s ({skad_sano}).")
    if S.dopelniony:
        ile = len(segm[0]) if segm else "mniej niż 16"
        print(f"Uwaga: za mało klatek brzucha na pełny segment (potrzeba 46, jest {n_b}). Segment ma {ile} z 16 klatek, "
              "resztę dopełniono zerami. Wynik jest tylko orientacyjny.")
    print(f"\nMasa W DNIU PORODU wg modelu Sano: {gramy(S.masa_g)} g" + ("  ← orientacyjnie" if S.dopelniony else ""))
    if n_s > 1:
        print(f"  oceny poszczególnych segmentów: {gramy(min(S.oceny_g))}–{gramy(max(S.oceny_g))} g")
    print(f"Masa TERAZ (AC z nnU-Netu, wzór Campbella): {o.komunikat}")
    print(f"Na czym uczył się model: masa urodzeniowa {gramy(MASA_UR_UCZACA)} ± {SD_UCZACA} g, "
          "ciąże donoszone (38,3 ± 1,5 tc.), badanie mniej niż 24 h przed porodem.")
    if o.wiarygodne and o.tydzien is not None and o.tydzien < 37:
        print(f"Wg AC to ok. {po_polsku(o.tydzien)} tc.: model wychodzi poza dane, na których się uczył, "
              "a mimo to podaje liczbę bez cienia wątpliwości. Model nie wie, kiedy nie wie.")
    print("Prawdziwej masy urodzeniowej tego dziecka nie znamy: to nie jest walidacja.")

    # Masa dziś a wynik modelu Sano (masa w dniu porodu), na tle zbioru uczącego modelu
    paski = []
    if o.masa_g is not None and o.wiarygodne:
        paski.append(("masa teraz\n(AC z nnU-Netu, Campbell)", o.masa_g, "#4E79A7"))
    eksperci = dane.ac_referencyjne(REFERENCJA, SWEEP.nazwa) if REFERENCJA is not None else []
    if eksperci:
        ac_eks = float(np.mean(eksperci))
        paski.append((f"masa teraz\n(AC ekspertów {ac_eks:.0f} mm, Campbell)", biometria.masa_campbell_g(ac_eks), "#A0BCD8"))
    paski.append(("masa w dniu porodu\nwg modelu Sano" + (" *" if S.dopelniony else ""), S.masa_g, "#E07B39"))
    y = np.arange(len(paski))[::-1]
    fig, os_ = plt.subplots(figsize=(10, 1.3 + 0.75 * len(paski)))
    os_.axvspan(MASA_UR_UCZACA - SD_UCZACA, MASA_UR_UCZACA + SD_UCZACA, color="0.9", zorder=0)
    os_.axvline(MASA_UR_UCZACA, color="0.6", lw=1, ls="--", zorder=0)
    os_.barh(y, [p[1] for p in paski], color=[p[2] for p in paski], height=0.55, zorder=2)
    for yi, (_, wartosc, _) in zip(y, paski):
        os_.text(wartosc + 50, yi, f"{gramy(wartosc)} g", va="center", fontsize=10, zorder=3)
    os_.text(MASA_UR_UCZACA, len(paski) - 0.45, f"zbiór uczący modelu (dzień porodu):\nmasa ur. {gramy(MASA_UR_UCZACA)} ± {SD_UCZACA} g",
             ha="center", va="bottom", fontsize=8, color="0.35",
             bbox=dict(facecolor="0.9", edgecolor="none", pad=1), zorder=1)
    os_.set_yticks(y, [p[0] for p in paski], fontsize=9)
    os_.set_ylim(-0.6, len(paski) + 0.15)
    os_.set_xlim(0, 4900); os_.set_xlabel("masa [g]")
    os_.spines[["top", "right"]].set_visible(False)
    os_.set_title("Masa dziś a wynik modelu Sano (masa w dniu porodu)" + ("   (* segment dopełniony, orientacyjnie)" if S.dopelniony else ""),
                  fontsize=11)
    fig.tight_layout()
    plt.show()

## Podsumowanie

**Trzy kroki, trzy modele:**
1. **nnU-Net z ACOUSLIC-AI (Holandia): działa, masa teraz.** Sam przegląda całe nagranie „na ślepo” (na żywo co 4. klatkę, wcześniej co 2.) i znajduje klatkę z brzuchem płodu. Z tej klatki liczymy obwód brzucha i szacujemy dzisiejszą masę oraz wiek ciążowy odpowiadający temu AC; obok widać pomiar ekspertów. Ale to model holenderski.
2. **FUVAI (Sano + WUM): polski, z publicznymi wagami, działa częściowo.** Na nagraniach „na ślepo” jest poza swoją domeną: rzadko jest pewny, że widzi brzuch, a klatkę wybieramy własną regułą. Na części nagrań nie znajduje brzucha wcale; tam, gdzie znajduje, nnU-Net nie zawsze widzi w tej klatce brzuch, a AC bywa daleko od pomiaru ekspertów. Masa tylko z AC, bo w tych danych nie ma głowy ani kości udowej.
3. **Model Sano (TabAttention, Sano + WUM): polski, publiczny checkpoint modelu bazowego, szacuje masę w dniu porodu, ale nasze nagrania są poza jego danymi.** Działa na naszym nagraniu w ułamku sekundy na zwykłym procesorze. Uczono go jednak tylko na badaniach wykonanych mniej niż 24 h przed porodem, więc na nagraniach z ok. 28–34 tc. wychodzi poza dane, na których się uczył, i każdemu dziecku podaje liczbę typową dla noworodka (3,6–4,0 kg) bez cienia wątpliwości. Model tego nie zasygnalizuje: nie wie, kiedy nie wie.

Wszystko na zwykłym laptopie: na żywo ok. 25–35 s nnU-Net, ok. 8 s FUVAI i ok. 1–2 s model Sano na nagraniu 840-klatkowym.

**Co z tego wynika (uczciwie):** nie pokazaliśmy narzędzia lepszego od Hadlocka. Pokazaliśmy trzy rzeczy:
1. **Automatyzacja na trudnych danych.** Model sam znajduje brzuch w 840 klatkach nagrania „na ślepo” (tania głowica, smartfon, nowicjusz); AC wychodzi kilka–kilkanaście mm od pomiaru ekspertów. Hadlock wymaga doświadczonego sonografisty.
2. **Najlepszy był model obraz + pomiary lekarza.** W pracy TabAttention sam obraz mylił się bardziej niż klinicyści (289 g vs 205 g), a obraz + pomiary biometryczne mniej (170 g). AI nie zastępuje pomiarów lekarza, tylko je wzmacnia.
3. **Model nie wie, kiedy nie wie.** Uczony na dniu porodu, płodowi z 28. tc. podaje 3,6–4 kg bez wahania. Każdemu produktowi AI warto zadać pytanie: „na jakich danych to uczono?”.

**Czego brakuje:**
- publicznie dostępnych wag pełnych modeli, które w publikacjach były lepsze od klinicystów: BabyNet++ (5,1% błędu vs 5,4%) i TabAttention z pomiarami biometrycznymi (170 g vs 205 g); opisano je w pracach, ale wytrenowanych wag nie udostępniono publicznie,
- nagrań z polskich szpitali **ze znaną masą urodzeniową**, także sprzed terminu porodu: na nich te modele można by sprawdzić, a model Sano douczyć. FUVAI i model Sano uczono na standardowych nagraniach szpitalnych, nie na nagraniach „na ślepo”.

Kod i prace źródłowe: [BabyNet++](https://github.com/SanoScience/BabyNetPlusPlus) (Płotka i in., *Computers in Biology and Medicine* 2023), [TabAttention](https://github.com/SanoScience/Tab-Attention) (Grzeszczyk, Płotka i in., MICCAI 2023), [FUVAI](https://github.com/SanoScience/FUVAI) (Płotka i in., *Physics in Medicine & Biology* 2022). Dziękujemy zespołom Sano i WUM za publicznie udostępnione prace, kod i wagi; kodu ani wag autorów nie rozpowszechniamy.

> Materiał edukacyjny. Nie jest wyrobem medycznym i nie służy do decyzji klinicznych.